<a href="https://colab.research.google.com/github/zsw-issac/ssemarket_LLM/blob/main/RAG_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install chromadb

创建向量库,并测试检索结果,与后面的区别是,这里会输出帖子的回答,不是生成的自然语言

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
import json

# ========== 1. 加载数据 ==========
with open('cleaned_result.json', 'r', encoding='utf-8') as f:
    cleaned_data = json.load(f)

print(f"加载了 {len(cleaned_data)} 条文档")

# ========== 2. 初始化嵌入模型（本地） ==========
# 这个模型会把文本变成向量，完全免费，本地运行
embed_model = SentenceTransformer('BAAI/bge-large-zh-v1.5')  # 中文效果好
print("嵌入模型加载完成")

# ========== 3. 创建向量数据库 ==========
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 创建集合（类似数据库的表）
collection = chroma_client.get_or_create_collection(
    name="sse_posts",
    metadata={"hnsw:space": "cosine"}  # 使用余弦相似度
)

# ========== 4. 向量化并存储 ==========
def index_documents(data_list):
    """将文档批量存入向量数据库"""

    # 准备数据
    ids = [str(item['post_id']) for item in data_list]
    documents = [item['formatted_text'] for item in data_list]
    metadatas = [item['metadata'] for item in data_list]

    # 生成向量（这一步最耗时）
    print("正在生成向量嵌入...")
    embeddings = embed_model.encode(documents, show_progress_bar=True)

    # 将id,文本转化后的向量,原始文本,元数据(方便溯源)存入数据库
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        documents=documents,
        metadatas=metadatas
    )

    print(f"✅ 成功索引 {len(data_list)} 条文档")

# 执行索引
index_documents(cleaned_data)

# ========== 5. 检索函数 ==========
def search(query, top_k=5):
    """根据问题检索相关文档"""

    # 把问题转成向量
    query_embedding = embed_model.encode([query])

    # 向量相似度搜索
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    return results

# ========== 6. 测试检索 ==========
test_query = "保研需要准备什么"
results = search(test_query, top_k=3)

print(f"\n查询: {test_query}")
print("="*80)
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), 1):
    print(f"\n【结果 {i}】 相似度: {1-dist:.3f}")
    print(f"标题: {meta['title']}")
    print(f"时间: {meta['post_time']}")
    print(f"内容预览: {doc[:200]}...")

加载了 3401 条文档


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


嵌入模型加载完成
正在生成向量嵌入...


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

✅ 成功索引 3401 条文档

查询: 保研需要准备什么

【结果 1】 相似度: 0.708
标题: 保研求助
时间: 2025年12月08日
内容预览: 标题: 保研求助
发布时间: 2025年12月08日
作者: 井芹仁仁菜
内容: 想打听一下保研的流程。
鼠鼠今年大三，绩点还过得去，想申请保研。有没有学长学姐分享一下保研的流程，有哪些重要时间节点，要做什么准备工作。
评论区: [花导🌸~: 主要就是准备材料和复习，然后在明年5月份的时候关注各个学校的夏令营就好了，下面是我之前给自己列的一个很简单的清单![1000012944.jpg](http...

【结果 2】 相似度: 0.700
标题: 想问下保研的流程是怎样的呢
时间: 2024年09月12日
内容预览: 标题: 想问下保研的流程是怎样的呢
发布时间: 2024年09月12日
作者: 花导🌸~
内容: rt，想问下保研的话是需要自己向学院申请资格，然后再进行一些夏令营的准备吗，然后本科中的一些竞赛和科研经历会很重要吗？（我虽然有参加科研或者比赛，但都很水，也没什么成果
评论区: [不想成为大水怪: 报夏令营就行，保研申请后面老师一定会发通知的，科研和比赛能有最好要有，现在还来得及，能有paper再加...

【结果 3】 相似度: 0.694
标题: 保研与考研经验分享 Q&A 指南
时间: 2024年10月22日
内容预览: 标题: 保研与考研经验分享 Q&A 指南
发布时间: 2024年10月22日
作者: ytj头号铁粉
内容: # 保研与考研经验分享 Q&A 指南
## 1. 保研基础准备问题
Q: 保研需要提前准备什么？
A: 
- 学习成绩（GPA）：保持在专业前20%-30%
- 英语水平：四六级成绩，部分学校要求雅思/托福
- 科研经历：参与导师项目或大创项目
- 竞赛经历：ACM、数学建模等高含金量竞赛...


加入deepseek api, 可以根据原文本生成答案

In [3]:
from google.colab import userdata
deepseek_api=userdata.get('deepseek_api')

In [4]:
from openai import OpenAI

# ========== 1. 初始化 DeepSeek 客户端 ==========
client = OpenAI(
    api_key=deepseek_api, # 替换为你的真实 Key
    base_url="https://api.deepseek.com"
)

In [5]:
def get_ai_answer(query, top_k=5):
    """
    检索并生成答案的完整流程
    """
    # 第一步：调用你之前的 search 函数获取相关文档
    search_results = search(query, top_k=top_k)

    # 第二步：将检索到的文档拼接成背景知识 (Context)
    # 把 documents[0] 里的多个文本块合成一段话
    context_list = search_results['documents'][0]
    context_text = "\n---\n".join([f"资料{i+1}: {doc}" for i, doc in enumerate(context_list)])

    # 第三步：构建给 AI 的 Prompt
    # 这里的角色设定非常重要，决定了 AI 是否会“乱说话”
    system_prompt = "你是一个校园资讯助手。请根据提供的参考资料回答用户问题。如果资料中没有相关信息，请诚实回答不知道，不要胡乱猜测。"

    user_prompt = f"""
参考资料：
{context_text}

用户问题：{query}

请基于上述资料，给出一个条理清晰的回答：
"""

    # 第四步：调用 API 获取生成结果
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream=False # 如果需要打字机效果，可以改为 True
    )

    return response.choices[0].message.content

# ========== 3. 测试 RAG 效果 ==========
question = "保研需要准备什么"
final_answer = get_ai_answer(question)

print(f"\n🚀 AI 基于本地数据库的回答：\n")
print("-" * 50)
print(final_answer)
print("-" * 50)


🚀 AI 基于本地数据库的回答：

--------------------------------------------------
根据提供的参考资料，保研的准备工作主要分为**硬性条件准备**、**申请材料准备**、**知识能力复习**和**流程策略规划**四个方面。以下是详细的梳理：

### 一、 硬性条件准备（长期积累）
这些是保研资格和竞争力的基础，需要尽早准备并保持。
1.  **学习成绩（GPA与排名）**：这是最重要的指标之一。需要保持较高的绩点，专业排名尽量靠前（例如资料中提到“中九rk3”等说法，指排名百分比）。排名是许多学校（尤其是“强com”学院）筛选入营资格的关键。
2.  **英语水平**：通过大学英语四级是基本要求，六级成绩**越高越好**（建议500分以上）。部分顶尖项目可能对雅思/托福有要求。
3.  **科研经历**：参与导师项目、大学生创新创业项目（大创）或实验室工作。**重要的是能清晰阐述自己在项目中的贡献、所用技术和项目逻辑**，而不一定需要发表顶会论文（大多数同学都没有）。
4.  **竞赛经历**：参加高含金量竞赛如ACM、数学建模等，可以获得重要加分。

### 二、 申请材料准备（大三下及之前）
用于提交给目标院校，需要精心制作。
1.  **必备证明**：
    *   **成绩单**：可从学校系统下载带电子章版本。
    *   **排名证明**：注意区分“教务系统排名”和“按保研算法计算的排名”，可根据情况选择有利的提交。
2.  **个人文书**：
    *   **个人简历**：建议使用PDF格式，排版简洁清晰，重点突出科研、竞赛和项目经历。避免大量篇幅描述学生工作或志愿活动。
    *   **个人陈述/研究计划**：阐述个人经历、研究兴趣和未来规划。
    *   **自我介绍PPT**：用于面试，内容应逻辑清晰，避免全文字和错别字。建议同时准备PDF版本以防格式错乱。
3.  **注意事项**：所有材料提交时注意**文件命名规范**（如：学校-姓名-申请类型-材料名称.pdf），确保正式、整洁。

### 三、 知识与能力复习（夏令营前集中准备）
针对夏令营和预推免的考核。
1.  **专业课知识**：重点复习**408核心课程**（数据结构、操作系统、计算机网络、计算机组成原理）及

In [17]:
question = "高数老师哪个好"
final_answer = get_ai_answer(question)

print(f"\n🚀 AI 基于本地数据库的回答：\n")
print("-" * 50)
print(final_answer)
print("-" * 50)


🚀 AI 基于本地数据库的回答：

--------------------------------------------------
根据提供的参考资料，以下是学生们推荐的高数老师，分为**网课老师**和**校内老师**两类：

### 一、网课/线上老师推荐
主要来自B站等平台，适合课后自学或补充学习。

1. **宋浩老师**
   * **推荐理由**：在多个求推荐网课的帖子中被反复提及，是热门推荐。
   * **特点**：课程受众广，适合基础学习。

2. **框框老师**
   * **推荐理由**：有学生推荐用于期末速成，并取得了不错的效果。
   * **特点**：被提及适合考前突击。

3. **一高数（B站UP主）**
   * **推荐理由**：有学生表示看了其速成课后排名提升显著。
   * **特点**：课程效果受到肯定。

4. **孔祥仁老师（B站）**
   * **推荐理由**：有评论认为他“讲得特别细”。
   * **注意**：课程顺序可能与某些教材版本（如北大版）不一致。

5. **CMC**
   * **推荐理由**：被推荐用于“拔高”学习。
   * **特点**：适合在基础之上进行提升。

### 二、校内老师推荐
适合旁听或选课，主要基于学生的课堂体验。

1. **杨朝霞老师**
   * **推荐理由**：受到高度赞扬，被形容为“讲的深入浅出，条理清晰，浅显易懂”。结课时学生自发鼓掌、合影。
   * **特点**：教学认真，但作业较多。不过对于旁听者而言，这恰好不是问题。

2. **顾会玲老师**
   * **推荐理由**：多次被提及，评价极高，如“顶级好老师”、“喂饭型讲课”。板书详细，课堂体验好。
   * **特点**：讲解细致，板书多，能让学生课下节省很多功夫。

3. **周yr老师**
   * **推荐理由**：板书多且清晰，讲解不错。
   * **注意**：有评论提到其声音可能有点小、沙哑。

### 总结与建议
* **如果你需要网课自学**：可以从**宋浩**、**框框**或**一高数**等热门选择入手，根据自身是打基础还是求速成的需求来决定。
* **如果你想寻找校内老师旁听**：**杨朝霞老师**和**顾会玲老师**是资料中口碑最好的两位，你可以进一步查询她们的课程时间安排